# Отчёт: Корпоративный ассистент АО «Осинка» — Продвинутый RAG

Полный отчёт по результатам ДЗ. Каждый блок (`# %%`) — отдельная ячейка.
Запуск: `uv run python notebooks/report.py` или ячейками в VS Code.

In [1]:
# Инициализация: импорты и конфигурация

import json
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from corporate_assistant.config import CHUNK_SIZE, CHUNK_OVERLAP, COLLECTION_NAME, RAW_PDFS, RAW_WIKI
from corporate_assistant.parsing import load_all_chunks, parse_pdf

print(f"Проект: {ROOT}")
print(f"Чанки: size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}")
print(f"Qdrant коллекция: {COLLECTION_NAME}")

/home/rtg/github/gritsulyak/llm-learn-tmp-2026/lessons/06_l16_advanced_rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Проект: /home/rtg/github/gritsulyak/llm-learn-tmp-2026/lessons/06_l16_advanced_rag
Чанки: size=600, overlap=60
Qdrant коллекция: corporate_assistant_2026


## 1. Парсинг PDF: метаданные страниц

In [2]:
print("=" * 70)
print("1. Извлечение метаданных страниц из PDF")
print("=" * 70)

for pdf_path in sorted(RAW_PDFS.glob("*.pdf")):
    pages = parse_pdf(pdf_path)
    print(f"\n{pdf_path.name} ({len(pages)} контентных страниц):")
    for p in pages:
        print(f"  phys={p.physical_page:2d}  printed={str(p.page):>4s}  header=\"{p.header}\"")

1. Извлечение метаданных страниц из PDF

Инструкция_по_больничным_листам.pdf (4 контентных страниц):
  phys= 1  printed=   1  header="Общие положения"
  phys= 2  printed=   2  header="Оформление больничного"
  phys= 3  printed=   3  header="Оплата больничного"
  phys= 4  printed=   4  header="Выход на работу"

Инструкция_по_отпускам.pdf (6 контентных страниц):
  phys= 3  printed=   3  header="Общие положения"
  phys= 4  printed=   4  header="Продолжительность ежегодного отпуска"
  phys= 5  printed=   5  header="Заявление на отпуск"
  phys= 6  printed=   6  header="Учебный отпуск"
  phys= 7  printed=   7  header="Отпуск без сохранения заработной платы"
  phys= 8  printed=   8  header="Памятка: частые вопросы"

Политика_удаленной_работы_2026.pdf (3 контентных страниц):
  phys= 1  printed=   1  header="Форматы работы"
  phys= 2  printed=   2  header="Исключения и Remote First"
  phys= 3  printed=   3  header="Оборудование и связь"

Положение_о_командировках.pdf (4 контентных страниц):
  p

### Ключевой момент: сдвинутая нумерация

В `Положение_о_командировках.pdf` нумерация сдвинута на 2 (титул + оглавление):
- физическая страница 5 → печатный номер «стр. 3»
- парсер берёт печатный номер для цитирования → ссылки соответствуют
  внутренней нумерации документа, а не физическому порядку листов.

In [3]:
print("=" * 70)
print("2. Пример: Положение_о_командировках.pdf — сдвинутая нумерация")
print("=" * 70)

trip_pdf = RAW_PDFS / "Положение_о_командировках.pdf"
trip_pages = parse_pdf(trip_pdf)
for p in trip_pages:
    offset_note = " ← физ. стр.≠печатный" if p.physical_page != p.page else ""
    print(f"  физ.{p.physical_page} → печ. {p.page}  header=\"{p.header}\"{offset_note}")

2. Пример: Положение_о_командировках.pdf — сдвинутая нумерация
  физ.3 → печ. 1  header="Общие положения" ← физ. стр.≠печатный
  физ.4 → печ. 2  header="Оформление командировки" ← физ. стр.≠печатный
  физ.5 → печ. 3  header="Расходы и суточные" ← физ. стр.≠печатный
  физ.6 → печ. 4  header="Отчет по командировке" ← физ. стр.≠печатный


## 2. Парсинг Wiki и статистика чанков

In [4]:
print("=" * 70)
print("3. Wiki-статьи")
print("=" * 70)

for html_path in sorted(RAW_WIKI.glob("*.html")):
    from corporate_assistant.parsing import parse_html
    chunks = parse_html(html_path)
    print(f"\n{html_path.name}:")
    for c in chunks:
        print(f"  header=\"{c.metadata['header']}\"  ({len(c.text)} символов)")

3. Wiki-статьи

Дресс-код.html:
  header="Базовый дресс-код"  (323 символов)
  header="Исключения"  (208 символов)

Компенсация_обедов.html:
  header="Условия компенсации"  (383 символов)
  header="Как потратить"  (123 символов)

Корпоративные_пароли.html:
  header="Требования к паролю"  (309 символов)
  header="Смена пароля и двухфакторная аутентификация"  (298 символов)
  header="Доступ к системам"  (251 символов)

Проект_Atlas.html:
  header="О проекте"  (316 символов)
  header="Статус и системы"  (249 символов)


In [5]:
print("=" * 70)
print("4. Статистика чанков")
print("=" * 70)

chunks = load_all_chunks()
print(f"Всего чанков: {len(chunks)}")

pdf_chunks = [c for c in chunks if c.doc_type == "pdf"]
wiki_chunks = [c for c in chunks if c.doc_type == "wiki"]
print(f"  PDF:  {len(pdf_chunks)}")
print(f"  Wiki: {len(wiki_chunks)}")
print(f"  Средний размер чанка: {sum(len(c.text) for c in chunks) / len(chunks):.0f} символов")

4. Статистика чанков
Всего чанков: 29
  PDF:  20
  Wiki: 9
  Средний размер чанка: 378 символов


## 3. Система цитирования (Prompt Engineering)

Три уровня защиты от галлюцинаций источников:
1. **Системный промпт** — строгие правила формата ссылок.
2. **Список доступных ссылок** — модель может использовать ТОЛЬКО строки из этого списка.
3. **Grounding** (постобработка) — каждая ссылка сверяется с реальными метаданными чанков.

In [6]:
print("=" * 70)
print("5. Системный промпт (фрагмент)")
print("=" * 70)

from corporate_assistant.prompts import SYSTEM_PROMPT
print(SYSTEM_PROMPT[:600])
print("...")

5. Системный промпт (фрагмент)
Ты — корпоративный ассистент компании АО «Осинка». Твоя задача — отвечать на вопросы сотрудников.

Строгие правила:
1. Отвечай строго и только на основе предоставленного контекста. Если ответа нет в контексте, напиши ровно одну фразу: «Я не знаю».
2. Не выдумывай факты и не используй собственные знания, которых нет в контексте.
3. Подтверждай каждый факт ссылкой на источник.
4. Ссылку ставь сразу после утверждения, которое она подтверждает.
5. Если используешь информацию из нескольких источников, приведи ссылку на каждый из них.

Ссылки:
- В сообщении пользователя есть «Список доступных ссылок
...


In [7]:
print("=" * 70)
print("6. Формат контекста для модели")
print("=" * 70)

from corporate_assistant.prompts import format_context, user_message, source_label, allowed_citations

# Симуляция: берём 2 чанка как пример
example_nodes = []
for c in chunks[:2]:
    class FakeNode:
        def __init__(self, chunk):
            self.text = chunk.text
            self.metadata = chunk.metadata
    class FakeNodeWithScore:
        def __init__(self, chunk):
            self.node = FakeNode(chunk)
            self.score = 0.85
    example_nodes.append(FakeNodeWithScore(c))

ctx = format_context(example_nodes)
print("Контекст:")
print(ctx[:500])
print("...\n")

citations = allowed_citations(example_nodes)
print("Список доступных ссылок:")
for i, c in enumerate(citations, 1):
    print(f"  {i}. {c}")

6. Формат контекста для модели
Контекст:
Контекст 1 (Источник: Инструкция_по_больничным_листам.pdf, стр. 1):
Инструкция по больничным листам
Отдел HR • Внутренняя документация
Документ является официальной инструкцией компании АО «Осинка». Общие положения
Настоящая инструкция описывает порядок оформления и оплаты листков
нетрудоспособности (больничных) в компании АО «Осинка». Больничный лист оформляется в электронном виде через систему Фонда
социального страхования. Бумажные бланки не требуются. Сотрудник обязан в первый день болезни с
...

Список доступных ссылок:
  1. Инструкция_по_больничным_листам.pdf, стр. 1
  2. Инструкция_по_больничным_листам.pdf, стр. 2


## 4. Тестирование: прогон 6 вопросов

In [8]:
print("=" * 70)
print("7. Тестовые вопросы и ответы")
print("=" * 70)

from corporate_assistant.assistant import CorporateAssistant

assistant = CorporateAssistant(collection_name=COLLECTION_NAME)

TESTS = [
    ("1. Прямой вопрос (один факт)",
     "За сколько дней нужно писать заявление на отпуск?"),
    ("2. Печатный номер страницы (сдвинутая нумерация)",
     "Какой размер суточных выплачивается при командировке и каков лимит расходов на проживание?"),
    ("3. Агрегация из двух документов",
     "Я уезжаю в командировку, а сразу после неё планирую уйти в отпуск. Какие сроки подачи заявлений нужно соблюсти для каждого события?"),
    ("4. Wiki-статья",
     "Какие требования к корпоративным паролям и как часто их нужно менять?"),
    ("5. Out-of-domain (провокация)",
     "Какая зарплата у Senior Python Developer и какой годовой бонус?"),
    ("6. Заголовок раздела + вычисление по фактам одного файла",
     "Сотруднику со стажем 7 лет открыли больничный. Сколько процентов от заработка он получит?"),
]

results = []
for name, query in TESTS:
    print(f"\n{'─' * 70}")
    print(f"Вопрос: {name}")
    print(f"Запрос: {query}")
    print("─" * 70)
    answer = assistant.answer(query)
    print(f"Ответ:  {answer.answer}")
    print(f"Источники: {[s['metadata'].get('source') for s in answer.sources]}")
    print(f"Отброшенные ссылки: {answer.dropped_citations or 'нет'}")
    results.append({"name": name, "query": query, "answer": answer})

7. Тестовые вопросы и ответы
Embedding model: intfloat/multilingual-e5-small on cpu


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15392.65it/s]


──────────────────────────────────────────────────────────────────────
Вопрос: 1. Прямой вопрос (один факт)
Запрос: За сколько дней нужно писать заявление на отпуск?
──────────────────────────────────────────────────────────────────────


Ответ:  Не позднее чем за 14 календарных дней до начала отпуска [Инструкция_по_отпускам.pdf, стр. 8].
Источники: ['Инструкция_по_отпускам.pdf', 'Инструкция_по_отпускам.pdf', 'Инструкция_по_отпускам.pdf', 'Инструкция_по_отпускам.pdf', 'Инструкция_по_отпускам.pdf']
Отброшенные ссылки: нет

──────────────────────────────────────────────────────────────────────
Вопрос: 2. Печатный номер страницы (сдвинутая нумерация)
Запрос: Какой размер суточных выплачивается при командировке и каков лимит расходов на проживание?
──────────────────────────────────────────────────────────────────────


Ответ:  Суточные выплачиваются в размере 1000 рублей за каждый календарный день командировки [Положение_о_командировках.pdf, стр. 3]. Лимит расходов на проживание составляет не более 6000 рублей за ночь для Москвы и Санкт-Петербурга и не более 4000 рублей за ночь для остальных городов [Положение_о_командировках.pdf, стр. 3].
Источники: ['Положение_о_командировках.pdf', 'Положение_о_командировках.pdf', 'Положение_о_командировках.pdf', 'Положение_о_командировках.pdf', 'Инструкция_по_отпускам.pdf']
Отброшенные ссылки: нет

──────────────────────────────────────────────────────────────────────
Вопрос: 3. Агрегация из двух документов
Запрос: Я уезжаю в командировку, а сразу после неё планирую уйти в отпуск. Какие сроки подачи заявлений нужно соблюсти для каждого события?
──────────────────────────────────────────────────────────────────────


Ответ:  Заявление на командировку подается через HR-портал не позднее чем за 5 рабочих дней до даты выезда [Положение_о_командировках.pdf, стр. 2]. Заявление на отпуск подается не позднее чем за 14 календарных дней до начала отпуска [Инструкция_по_отпускам.pdf, стр. 8].
Источники: ['Инструкция_по_отпускам.pdf', 'Инструкция_по_отпускам.pdf', 'Инструкция_по_отпускам.pdf', 'Положение_о_командировках.pdf', 'Положение_о_командировках.pdf']
Отброшенные ссылки: нет

──────────────────────────────────────────────────────────────────────
Вопрос: 4. Wiki-статья
Запрос: Какие требования к корпоративным паролям и как часто их нужно менять?
──────────────────────────────────────────────────────────────────────


Ответ:  Корпоративные пароли должны содержать не менее 12 символов и включать буквы в верхнем и нижнем регистре, цифры и специальные символы [Корпоративные пароли и доступы]. Пароль должен быть сменен не реже одного раза в 90 дней [Корпоративные пароли и доступы].
Источники: ['Корпоративные пароли и доступы', 'Корпоративные пароли и доступы', 'Корпоративные пароли и доступы', 'Политика_удаленной_работы_2026.pdf', 'Дресс-код компании']
Отброшенные ссылки: нет

──────────────────────────────────────────────────────────────────────
Вопрос: 5. Out-of-domain (провокация)
Запрос: Какая зарплата у Senior Python Developer и какой годовой бонус?
──────────────────────────────────────────────────────────────────────


Ответ:  Я не знаю
Источники: ['Политика_удаленной_работы_2026.pdf', 'Политика_удаленной_работы_2026.pdf', 'Политика_удаленной_работы_2026.pdf', 'Инструкция_по_отпускам.pdf', 'Политика_удаленной_работы_2026.pdf']
Отброшенные ссылки: нет

──────────────────────────────────────────────────────────────────────
Вопрос: 6. Заголовок раздела + вычисление по фактам одного файла
Запрос: Сотруднику со стажем 7 лет открыли больничный. Сколько процентов от заработка он получит?
──────────────────────────────────────────────────────────────────────


Ответ:  Сотрудник со стажем 7 лет будет получать пособие в размере 80% от среднего заработка [Инструкция_по_больничным_листам.pdf, стр. 3].
Источники: ['Инструкция_по_больничным_листам.pdf', 'Инструкция_по_больничным_листам.pdf', 'Инструкция_по_больничным_листам.pdf', 'Инструкция_по_отпускам.pdf', 'Инструкция_по_отпускам.pdf']
Отброшенные ссылки: нет


### Чат-режим: реальный прогон с источниками

То же, что делает `scripts/chat.py`: ответ + список источников с указанием
раздела и страницы.

In [9]:
print("=" * 70)
print("Чат-режим: реальный прогон")
print("=" * 70)

chat_query = "За сколько дней писать заявление на отпуск?"
print(f"> {chat_query}\n")
chat_answer = assistant.answer(chat_query)
print(chat_answer.answer)
print()
print("Источники:")
for s in chat_answer.sources:
    meta = s["metadata"]
    label = meta.get("source", "?")
    section = meta.get("header", "")
    page = meta.get("page")
    if meta.get("doc_type") == "wiki":
        print(f"  • {label} — раздел «{section}»")
    else:
        print(f"  • {label} — раздел «{section}», стр. {page}")
print()

Чат-режим: реальный прогон
> За сколько дней писать заявление на отпуск?



Не позднее чем за 14 календарных дней до начала отпуска [Инструкция_по_отпускам.pdf, стр. 8].

Источники:
  • Инструкция_по_отпускам.pdf — раздел «Памятка: частые вопросы», стр. 8
  • Инструкция_по_отпускам.pdf — раздел «Заявление на отпуск», стр. 5
  • Инструкция_по_отпускам.pdf — раздел «Заявление на отпуск», стр. 5
  • Инструкция_по_отпускам.pdf — раздел «Продолжительность ежегодного отпуска», стр. 4
  • Инструкция_по_отпускам.pdf — раздел «Отпуск без сохранения заработной платы», стр. 7



## 5. Анализ результатов

In [10]:
print("=" * 70)
print("8. Итоговая таблица")
print("=" * 70)

print(f"\n{'Тест':<50} {'Результат'}")
print("─" * 65)
for r in results:
    a = r["answer"]
    status = "PASS" if not a.dropped_citations else f"DROPPED {len(a.dropped_citations)}"
    print(f"{r['name']:<50} {status}")

8. Итоговая таблица

Тест                                               Результат
─────────────────────────────────────────────────────────────────
1. Прямой вопрос (один факт)                       PASS
2. Печатный номер страницы (сдвинутая нумерация)   PASS
3. Агрегация из двух документов                    PASS
4. Wiki-статья                                     PASS
5. Out-of-domain (провокация)                      PASS
6. Заголовок раздела + вычисление по фактам одного файла PASS


### RAGAS-метрики (faithfulness, context_precision, context_recall)

Оценка пайплайна библиотекой RAGAS на 5 вопросах с эталонными ответами.
Судья и эмбеддинги — локальные (Ollama `qwen2.5:3b`). Результат кэшируется
в `results/ragas_scores.json`; при наличии файла выводятся сохранённые значения.

In [11]:
print("=" * 70)
print("10. RAGAS-метрики")
print("=" * 70)

import json as _json
from pathlib import Path as _Path

RAGAS_JSON = ROOT / "results" / "ragas_scores.json"
if RAGAS_JSON.exists():
    ragas_scores = _json.loads(RAGAS_JSON.read_text(encoding="utf-8"))
    print(f"(загружено из {RAGAS_JSON.relative_to(ROOT)})")
else:
    print("Пересчёт метрик (может занять ~5 мин)...")
    sys.argv = ["evaluate_ragas.py"]
    import runpy

    runpy.run_path(str(ROOT / "scripts" / "evaluate_ragas.py"), run_name="__main__")
    ragas_scores = _json.loads(RAGAS_JSON.read_text(encoding="utf-8"))

for metric, values in ragas_scores.items():
    mean = sum(float(v) for v in values) / len(values)
    print(f"  {metric:<40} {mean:.3f}")

10. RAGAS-метрики
(загружено из results/ragas_scores.json)
  faithfulness                             0.500
  llm_context_precision_with_reference     0.851
  context_recall                           1.000


### Интерпретация RAGAS-метрик

- **context_recall = 1.0** — ретривер находит всю информацию, нужную для ответа
  (полнота контекста идеальная; в нашем случае это ожидаемо: база небольшая и
  эталонные вопросы взяты из неё).
- **context_precision ≈ 0.85** — большинство извлечённых чанков релевантны вопросу;
  остальное — «шумовые» чанки по соседней теме (ретривер честно возвращает топ-5).
- **faithfulness ≈ 0.5** — занижен не столько из-за ответов (они по фактам верны),
  сколько из-за слабости маленькой локальной модели как судьи: она неполно
  разбивает ответ на утверждения и часть подтверждает «не подтверждено». Для
  продакшена судьёй должна быть сильная модель (GPT-класса).

## 6. Анализ: влияние chunk_size на точность страниц

In [12]:
print("=" * 70)
print("9. Эксперимент: chunk_size vs точность (top-3 retrieval)")
print("=" * 70)

from corporate_assistant.parsing import load_all_chunks
from corporate_assistant.indexing import build_index
from corporate_assistant.prompts import source_label

TEST_QUERIES = [
    ("Заявление на отпуск подается не позднее чем за 14 дней",
     "Инструкция_по_отпускам.pdf", 5),
    ("Суточные при командировке составляют 1000 рублей",
     "Положение_о_командировках.pdf", 3),
    ("Пароль должен содержать не менее 12 символов",
     "Корпоративные пароли и доступы", None),
    ("Удаленная работа доступна два дня в неделю",
     "Политика_удаленной_работы_2026.pdf", 1),
    ("Больничный при стаже 8 лет оплачивается 100%",
     "Инструкция_по_больничным_листам.pdf", 3),
]

for size in [300, 600, 1200]:
    coll_name = f"test_chunk_{size}"
    all_chunks = load_all_chunks(chunk_size=size, chunk_overlap=size // 10)
    index = build_index(all_chunks, collection_name=coll_name, reset=True)
    retriever = index.as_retriever(similarity_top_k=3)

    correct = 0
    for query, expected_source, expected_page in TEST_QUERIES:
        nodes = retriever.retrieve(query)
        found = False
        for n in nodes:
            label = source_label(n.node.metadata)
            if expected_source in label:
                if expected_page is None or expected_page == n.node.metadata.get("page"):
                    found = True
                    break
        if found:
            correct += 1

    print(f"  chunk_size={size:5d}  →  {correct}/5")

print("\nВывод: chunk_size не влияет на корректность номера страниц,")
print("поскольку чанки нарезаются внутри страницы (страница → чанки).")

9. Эксперимент: chunk_size vs точность (top-3 retrieval)
Удаляю коллекцию 'test_chunk_300'...
Embedding model: intfloat/multilingual-e5-small on cpu


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17486.88it/s]

Чанков к индексации: 50


Generating embeddings:   0%|          | 0/50 [00:00<?, ?it/s]

...

Generating embeddings:   0%|          | 0/26 [00:00<?, ?it/s]

Generating embeddings:  38%|███▊      | 10/26 [00:00<00:00, 48.97it/s]

Generating embeddings:  77%|███████▋  | 20/26 [00:00<00:00, 52.21it/s]

Generating embeddings: 100%|██████████| 26/26 [00:00<00:00, 55.59it/s]

Индекс собран за 0.7 c: Qdrant collection='test_chunk_1200'
  chunk_size= 1200  →  5/5

Вывод: chunk_size не влияет на корректность номера страниц,
поскольку чанки нарезаются внутри страницы (страница → чанки).


## 7. Выводы

### Что работает
- **Умное чанкование значительно лучше наивного RAG**: чанки нарезаются внутри
  страницы с сохранением метаданных (`page`, `header`, `source`), поэтому каждый
  чанк самодостаточен — ретривер возвращает контекст, который уже знает, из какого
  документа, раздела и страницы он взят. В наивном RAG чанк — это просто кусок
  текста без привязки к документу, из-за чего контекст обрезается и номер страницы
  «размазывается». Здесь контекст точный, а ответы ассистента корректно ссылаются
  на источник (подтверждено прогонами в разделе 4).
- **Точная привязка к странице**: чанкинг внутри страницы гарантирует, что
  `page`/`header` не «размазываются» — номер страницы всегда корректен.
- **Печатный номер vs физический**: парсер берёт печатный номер из колонтитула,
  поэтому ссылки совпадают с внутренней нумерацией документа (важно для сдвинутых PDF).
- **Три барьера против галлюцинаций**: промпт + список ссылок + grounding — модель
  не может сослаться на несуществующий источник; выдуманные ссылки удаляются и логируются.

### Ограничения
- Маленькая модель (3B) на длинных ответах иногда «забывает» поставить ссылку
  на часть утверждений — grounding не придумывает ссылку за неё (честный результат).
- Модель может поставить ссылку на близкий по теме, но неверный чанк —
  постобработка по точному совпадению строк это допускает; остаток фиксируют
  метрики RAGAS (см. раздел 5): context_precision ≈ 0.85, faithfulness ≈ 0.5.
- faithfulness занижает маленькая локальная модель-судья: для продакшена оценку
  RAGAS нужно проводить сильной моделью (GPT-класса).
- Сканированные PDF требуют OCR; точность определения номера страниц падает.

## 8. Проблемы, с которыми столкнулись в процессе

### Долгое «думание» большой локальной модели
- Первой подключили `qwen3.5:9b` (reasoning-модель): на CPU простой вопрос с одним
  фактом обрабатывался ~28 секунд — модель «рассуждала» перед каждым ответом.
  Для интерактивного ассистента это неприемлемо.
- Решение: заменили на `qwen2.5:3b` — ответ приходит за 1–3 секунды, а качество
  цитирования на коротких фактах не хуже (см. раздел 4).

### Промптинг и галлюцинации ссылок
- Маленькая модель систематически «фантазировала» ссылки: на вопрос про пароли
  (wiki) сослалась на `[Инструкция_по_отпускам.pdf, стр. 5]` — источник, которого
  не было среди извлечённых чанков.
- Промпт сам по себе не решает проблему, поэтому пошли по трём уровням:
  1) строгий системный промпт; 2) «Список доступных ссылок» в сообщении пользователя;
  3) grounding — постобработка, удаляющая выдуманные ссылки (см. раздел 3).
- Итерации промпта: добавляли явный запрет писать `[Контекст N]` / `[Источник ...]`,
  требование вставлять строку ссылки дословно и примеры корректных ответов.

### Библиотеки и версии
- `qdrant-client` пришлось закрепить `<1.13` — более свежая версия ломала совместимость
  с установленным `llama-index-vector-stores-qdrant`.
- PDF-парсинг делали на PyMuPDF, а не pdfplumber: быстрее и точнее извлекает
  позиции слов (`get_text("words")`), что нужно для колонтитулов.
- reportlab для генерации базы знаний потребовал регистрации шрифта DejaVu
  (TTFont) — иначе кириллица в PDF не собирается.
- sentence-transformers на CPU: embedding-модель `multilingual-e5-small` грузится
  ~15–20 с, но индексация 29 чанков занимает <1 с.

### Специфичные проблемы RAG
- **Сдвинутая нумерация страниц**: титул + оглавление сдвигают печатный номер
  относительно физического. Решено парсингом колонтитула (`стр. N`).
- **Служебные листы** (титул/оглавление) без заголовка раздела захламляли базу —
  их отбрасываем по отсутствию верхнего колонтитула.
- **Чанк между страницами**: при нарезке «документ → чанки» один чанк мог склеить
  текст со стр. 4 и 5, и номер страницы становился неоднозначным. Решено нарезкой
  «страница → чанки» (см. раздел 6).